# RDFLib

# Сергушов Павел. ПМ22-4

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Курс лекций [Semantic Technologies for Developers](https://yadi.sk/d/SepS4JBLhDYy4Q), лекция №02
* https://rdflib.readthedocs.io/en/stable/gettingstarted.html
* https://rdflib.readthedocs.io/en/stable/intro_to_graphs.html
* https://rdflib.readthedocs.io/en/stable/apidocs/rdflib.html#rdflib.Graph
* https://rdflib.readthedocs.io/en/stable/apidocs/rdflib.html#module-rdflib.paths  
* https://www.w3.org/TR/sparql11-query/#propertypaths
      

In [1]:
!pip install rdflib

## Вопросы для совместного обсуждения

1\. Обсудите работу с пакетом RDFLib и его возможности: чтение RDF-графа, навигация, работа с URI и пространствами имен, сериализация. 

RDFLib - это библиотека для работы с RDF. Она позволяет создавать, анализировать и манипулировать RDF-графами.

In [2]:
from rdflib import Graph
from rdflib import Namespace
from rdflib import URIRef

Для примера будем использовать небольшой граф из следующей ячейки.

In [2]:
example = """
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:John a ex:Person ;
    ex:name "John Doe" ;
    ex:age "30"^^xsd:integer ;
    ex:knows ex:Jane .

ex:Jane a ex:Person ;
    ex:name "Jane Smith" ;
    ex:age "28" ;
    ex:worksAt ex:TechCompany .

ex:TechCompany a ex:Company ;
    ex:foundedIn "2010" .
"""

Чтобы создать граф на основе сериализованного RDF, мы создаем объект Graph и вызываем метод parse. Если данные находятся в файле, то его нужно открыть и передать дескриптор файла в качестве аргумента `file`.

In [3]:
g = Graph()
g.parse(data=example, format="n3")

<Graph identifier=N4f21197da7a24b319a5f9129bf0bf86f (<class 'rdflib.graph.Graph'>)>

URIRef (URI Reference) - это класс в RDFLib, представляющий полный URI. Используется для уникальной идентификации ресурсов в RDF.

In [4]:
subj = URIRef("http://example.org/John")

In [5]:
subj

rdflib.term.URIRef('http://example.org/John')

Основные методы для работы с графом:


* `objects(subject, predicate)` - возвращает все объекты для данного субъекта и предиката.
    
* `predicate_objects(subject)` - возвращает все пары предикат-объект для данного субъекта.

* `subject_predicates(object)` - возвращает все пары субъект-предикат для данного объекта.

* `subjects(predicate, object)` - возвращает все субъекты для данного предиката и объекта.

* `triples((subject, predicate, object))` - возвращает все триплеты, соответствующие заданному шаблону. Можно использовать None для обозначения "любого" значения.
    

Данные методы возвращают генераторы, поэтому по ним нужно проитерироваться, чтобы "увидеть" результат.

In [6]:
list(g.objects(subject=subj))

[rdflib.term.URIRef('http://example.org/Person'),
 rdflib.term.Literal('John Doe'),
 rdflib.term.Literal('30', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')),
 rdflib.term.URIRef('http://example.org/Jane')]

Namespace - это удобный способ создания URIRef'ов в определенном пространстве имен. Позволяет сократить запись длинных URI.

In [7]:
ex = Namespace("http://example.org/")
ex.John

rdflib.term.URIRef('http://example.org/John')

In [8]:
objs = list(g.objects(subject=ex.John))
objs

[rdflib.term.URIRef('http://example.org/Person'),
 rdflib.term.Literal('John Doe'),
 rdflib.term.Literal('30', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')),
 rdflib.term.URIRef('http://example.org/Jane')]

In [9]:
ex_literal = objs[2]
ex_literal

rdflib.term.Literal('30', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))

У `Literal` есть следующие полезные атрибуты:

* value: Возвращает значение литерала.
  
* datatype: Возвращает тип данных литерала (если указан).
  
* language: Возвращает языковой тег (если указан).

Метод `toPython` преобразует литерал в объект Python соответствующего типа.

In [10]:
ex_literal.datatype

rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')

In [11]:
ex_literal.language

In [12]:
ex_literal.value

30

In [13]:
ex_literal.toPython()

30

При навигации по графу можно использовать специальные операторы, определенные стандартом SPARQL (о нем поговорим в следующим раз).
Несколько примеров:
* / (оператор деления): В контексте RDFLib это не математическое деление, а способ создания пути в графе. Он означает "следовать по этому предикату, затем по следующему" (т.н. SequencePath)
* | (оператор объединения): Объединяет результаты двух выражений предикатов.

In [14]:
# Так можно получить значение возраста Jane, не получая соответствующий ей узел напрямую.
list(g.objects(subject=ex.John, predicate=ex.knows/ex.age))

[rdflib.term.Literal('28')]

Графы можно модифицировать. Основной метод - `add` - позволяет добавить новую тройку.

In [15]:
from rdflib import Literal, XSD

In [16]:
g.add((ex.John, ex.knows, ex.Maria))
g.add((ex.Maria, ex.age, Literal(30, datatype=XSD.integer)))

<Graph identifier=N4f21197da7a24b319a5f9129bf0bf86f (<class 'rdflib.graph.Graph'>)>

In [17]:
list(g.objects(subject=ex.John, predicate=ex.knows/ex.age))

[rdflib.term.Literal('28'),
 rdflib.term.Literal('30', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))]

Для сериализации существует соответствующий метод.

In [18]:
print(g.serialize(format="turtle"))

@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:John a ex:Person ;
    ex:age 30 ;
    ex:knows ex:Jane,
        ex:Maria ;
    ex:name "John Doe" .

ex:Jane a ex:Person ;
    ex:age "28" ;
    ex:name "Jane Smith" ;
    ex:worksAt ex:TechCompany .

ex:Maria ex:age 30 .

ex:TechCompany a ex:Company ;
    ex:foundedIn "2010" .




## Задачи для самостоятельного решения

<p class="task" id="1"></p>

1\. Используя пакет `rdflib`, создайте RDF-граф на основе представления в формате N-Triples из задания __01.4__. Выведите на экран количество троек в загруженном графе.

Воспользовавшись методом `all_nodes()`, получите множество узлов графа. Выясните, сколько из них являются URI (проверьте на тип `rdflib.term.URIRef`) и литералами (проверьте на тип `rdflib.term.Literal`). Для литеральных типов проанализируйте статистику по типам (datatype). При отсутствии значения считайте, что тип равен `http://www.w3.org/2001/XMLSchema#string`.

- [ ] Проверено на семинаре

In [2]:
from rdflib import Graph, URIRef, Literal
from collections import Counter

In [3]:
g = Graph()

# Каждая строка представляет собой одну тройку: (субъект, предикат, объект).
data = """<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Name> "Анна"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Surname> "Петрова"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Age> "35"^^<http://www.w3.org/2001/XMLSchema#integer> .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/LivesIn> <http://www.fa.ru/entity/ID-2> .
<http://www.fa.ru/entity/ID-2> <http://www.fa.ru/predicate/Name> "Зеленогорск"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/SpecializesIn> "пейзажная живопись"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Created> <http://www.fa.ru/entity/ID-3> .
<http://www.fa.ru/entity/ID-3> <http://www.fa.ru/predicate/Name> "Времена года"@ru .
<http://www.fa.ru/entity/ID-3> <http://www.fa.ru/predicate/ConsistsOf> "4" .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Uses> "различные техники" .
<http://www.fa.ru/entity/ID-4> <http://www.fa.ru/predicate/Name> "Весеннее пробуждение"@ru .
<http://www.fa.ru/entity/ID-4> <http://www.fa.ru/predicate/Technique> "импасто"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/FavoriteColor> "зеленый"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/FavoriteColor> "синий"@ru .
<http://www.fa.ru/entity/ID-5> <http://www.fa.ru/predicate/Name> "мастерская"@ru .
<http://www.fa.ru/entity/ID-5> <http://www.fa.ru/predicate/Address> "улица Лесная, дом 7"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Teaches> "мастер-классы"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Phone> "+7 (123) 456-78-90" ."""

In [ ]:
# Используем метод .parse() для загрузки данных в граф.
g.parse(data=data, format="nt")

print(f"Количество троек в графе: {len(g)}")

Количество троек в графе: 18


In [5]:
# Анализ узлов графа 
# Метод .all_nodes() возвращает итератор по всем *уникальным* узлам в графе
nodes = g.all_nodes()

uri_count = 0
literal_count = 0
literal_datatypes = []

In [ ]:
# Проходим в цикле по каждому уникальному узлу в графе.
for node in nodes:
    if isinstance(node, URIRef):
        uri_count += 1
    elif isinstance(node, Literal):
        literal_count += 1
        # У объекта Literal есть атрибут .datatype, который хранит URI его типа.
        datatype = node.datatype or URIRef("http://www.w3.org/2001/XMLSchema#string")
        literal_datatypes.append(str(datatype))

In [7]:
print(f"Всего уникальных узлов: {len(list(g.all_nodes()))}") 
print(f"Количество узлов типа URI: {uri_count}")
print(f"Количество узлов типа Literal: {literal_count}")

Всего уникальных узлов: 21
Количество узлов типа URI: 5
Количество узлов типа Literal: 16


In [ ]:
print("\nСтатистика по типам данных литералов:")
# Создаем словарь, где ключи - это типы данных,
# а значения - их количество.
# Метод .items() позволяет пройти по парам (ключ, значение) этого словаря.
for datatype, count in Counter(literal_datatypes).items():
    print(f"- {datatype}: {count}")


Статистика по типам данных литералов:
- http://www.w3.org/2001/XMLSchema#string: 15
- http://www.w3.org/2001/XMLSchema#integer: 1


<p class="task" id="2"></p>

2\. Создайте объект `URIRef`, которые описывает художницу Анну. Выведите на экран информацию о тройках, в которых Анна участвует как субъект. Выведите их на экран.

Решите задачу тремя способами:
* проитерировавшись по объекту графа и отобрав нужные тройки;
* при помощи метода `Graph.triples`;
* при помощи метода `Graph.predicate_objects`.

- [ ] Проверено на семинаре

In [9]:
# Чтобы найти в графе конкретный ресурс Анны,нам нужно создать объект URIRef, который в точности соответствует её URI в данных.
anna_uri = URIRef("http://www.fa.ru/entity/ID-1")

In [ ]:
# Способ 1: Прямая итерация по всему графу
# возвращает кортеж из трех элементов: (subject, predicate, object).
for s, p, o in g:
    if s == anna_uri:
        print(f"{s.n3()} {p.n3()} {o.n3()} .")

<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/LivesIn> <http://www.fa.ru/entity/ID-2> .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Teaches> "мастер-классы"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Created> <http://www.fa.ru/entity/ID-3> .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Surname> "Петрова"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/FavoriteColor> "синий"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/SpecializesIn> "пейзажная живопись"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Phone> "+7 (123) 456-78-90" .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Uses> "различные техники" .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Name> "Анна"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/FavoriteColor> "зеленый"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Age> "35"^^<http://www.w3.org/2001/XML

In [ ]:
# Способ 2: Использование метода Graph.triples()

# Мы вызываем метод .triples() с шаблоном (anna_uri, None, None).

for s, p, o in g.triples((anna_uri, None, None)):
    print(f"{s.n3()} {p.n3()} {o.n3()} .")

<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Name> "Анна"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Surname> "Петрова"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Age> "35"^^<http://www.w3.org/2001/XMLSchema#integer> .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/LivesIn> <http://www.fa.ru/entity/ID-2> .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/SpecializesIn> "пейзажная живопись"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Created> <http://www.fa.ru/entity/ID-3> .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Uses> "различные техники" .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/FavoriteColor> "зеленый"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/FavoriteColor> "синий"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Teaches> "мастер-классы"@ru .
<http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Phone> "+7 (

In [ ]:
# Способ 3: Использование метода Graph.predicate_objects()
print(f"Предикаты и объекты для субъекта {anna_uri.n3()}:")
# Мы вызываем метод .predicate_objects(), передавая ему искомый субъект.
# Метод возвращает генератор, который на каждой итерации выдает
# кортеж из двух элементов: (predicate, object).
for p, o in g.predicate_objects(subject=anna_uri):
    print(f"  {anna_uri.n3()} {p.n3()} {o.n3()} .")


--- Способ 3: Метод Graph.predicate_objects() ---
Предикаты и объекты для субъекта <http://www.fa.ru/entity/ID-1>:
  <http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Name> "Анна"@ru .
  <http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Surname> "Петрова"@ru .
  <http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Age> "35"^^<http://www.w3.org/2001/XMLSchema#integer> .
  <http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/LivesIn> <http://www.fa.ru/entity/ID-2> .
  <http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/SpecializesIn> "пейзажная живопись"@ru .
  <http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Created> <http://www.fa.ru/entity/ID-3> .
  <http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/Uses> "различные техники" .
  <http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/FavoriteColor> "зеленый"@ru .
  <http://www.fa.ru/entity/ID-1> <http://www.fa.ru/predicate/FavoriteColor> "синий"@ru .
  <http://www.fa.ru/entity

<p class="task" id="3"></p>

3\. Используя пакет `rdflib`, создайте RDF-граф на основе файла `data/university.n3`. Выведите на экран количество троек в загруженном графе.

Найдите и выведите на экран ректоров всех университетов при помощи метода `Graph.objects`.

- [ ] Проверено на семинаре

In [15]:
from rdflib import Graph, Namespace

In [4]:
g_univ = Graph()

g_univ.parse("/Users/andreyserba/Desktop/4_курс_1/Семантические_технологии/university.n3", format="n3")
print("Файл 'university.n3' успешно загружен.")

Файл 'university.n3' успешно загружен.


In [18]:
print(f"Количество троек в графе: {len(g_univ)}")

Количество троек в графе: 52


In [19]:
# Определяем пространства имен для удобства.
E = Namespace("http://www.fa.ru/entity/")
PD = Namespace("http://www.fa.ru/prop/direct/")

# Находим все URI, которые являются университетами.
universities = g_univ.subjects(predicate=PD.IsA, object=E.University)

print("Ректоры университетов:")
for uni_uri in universities:
    # Для каждого университета ищем его ректора.
    for rector_uri in g_univ.objects(subject=uni_uri, predicate=PD.rector):
        # Очищаем URI для красивого вывода.
        uni_name = str(uni_uri).replace(str(E), "")
        rector_name = str(rector_uri).replace(str(E), "")
        
        print(f"- Университет: {uni_name}, Ректор: {rector_name}")

Ректоры университетов:
- Университет: MoscowStateUniversity, Ректор: Viktor_Sadovnichiy
- Университет: BaumanMoscowStateTechnicalUniversity, Ректор: Anatoly_Aleksandrov
- Университет: SaintPetersburgStateUniversity, Ректор: Nikolay_Kropachev
- Университет: SpbPolytechnicUniversity, Ректор: Andrey_Rudskoy


<p class="task" id="4"></p>

4\. Для хранения информации о годах в данном графе была произведена процедура материализации по схеме 2. 

Создайте объект `URIRef`, которые описывает университет МГУ. Получите информацию о количестве студентов в МГУ за разные года, используя двухшаговую процедуру:
1. Найдите все объекты statement, связанные с МГУ соответствующим отношением;
2. Найдите все субъекты и значение квалификатора для каждого из statement. 

Представьте результат в виде словаря `msu_student_size`. Используйте только метод  `Graph.objects` для поиска информации. 

- [ ] Проверено на семинаре

In [21]:
# Инициализируем пустой словарь для хранения результатов,
# Указываем типы для наглядности: ключ - год (int), значение - кол-во студентов (int).
msu_student_size: dict[int, int] = {}

# Определяем все необходимые пространства имен из файла university.n3.
E = Namespace("http://www.fa.ru/entity/")
P = Namespace("http://www.fa.ru/prop/")
PS = Namespace("http://www.fa.ru/prop/statement/")
PQ = Namespace("http://www.fa.ru/prop/qualifier/")

msu_uri = E.MoscowStateUniversity

In [ ]:
# Мы ищем все тройки, где субъект - это МГУ, а предикат - p:studentSize.
# Объекты в этих тройках и будут искомыми узлами-утверждениями (s:a1, s:a2, s:a3).
for statement_node in g_univ.objects(subject=msu_uri, predicate=P.studentSize):
    print(f"  Найден узел-утверждение (statement): {statement_node.n3()}")

    year = None
    student_count = None
    
    # Находим значение (количество студентов).
    # Теперь субъектом поиска является сам узел-утверждение (statement_node).
    # Предикат - ps:studentSize, который хранит основное значение.
    for size_literal in g_univ.objects(subject=statement_node, predicate=PS.studentSize):
        # Результат - это литерал.
        student_count = size_literal.toPython()
        print(f"    - Найдена численность: {student_count}")

    # Находим квалификатор (год).
    # Субъект - тот же statement_node. Предикат - pq:atYear.
    for year_literal in g_univ.objects(subject=statement_node, predicate=PQ.atYear):
        #year = int(year_literal.toPython())
        year = year_literal.toPython().year
        print(f"    - Найден год: {year}")
        
    # добавляем их в наш итоговый словарь.
    if year is not None and student_count is not None:
        msu_student_size[year] = student_count


  Найден узел-утверждение (statement): <http://www.fa.ru/entity/statement/a1>
    - Найдена численность: 40000
    - Найден год: 2000
  Найден узел-утверждение (statement): <http://www.fa.ru/entity/statement/a2>
    - Найдена численность: 38000
    - Найден год: 2010
  Найден узел-утверждение (statement): <http://www.fa.ru/entity/statement/a3>
    - Найдена численность: 42000
    - Найден год: 2020


In [28]:
print("Итоговый словарь численности студентов МГУ по годам:")
print(msu_student_size)

Итоговый словарь численности студентов МГУ по годам:
{2000: 40000, 2010: 38000, 2020: 42000}


<p class="task" id="5"></p>

5\. При формировании предиката допускается использование специальных операторов, которые регламентированы стандартом SPARQL. Используя выражение SequencePath, получите значение численность университета за каждый год, не получая промежуточные узлы statement явно. Представьте результат в виде списка `msu_student_size`.

Используйте только метод  `Graph.objects` для поиска информации. 

- [ ] Проверено на семинаре

In [29]:
msu_student_size: list[int] = []

# Создаем объект SequencePath.
# Этот путь описывает маршрут от университета к численности студентов
path_to_size = P.studentSize / PS.studentSize

# Используем метод .objects() с нашим составным путем.
# rdflib автоматически выполнит всю работу по навигации.
results_generator = g_univ.objects(subject=msu_uri, predicate=path_to_size)


In [30]:
# Преобразуем генератор с литералами в список Python-чисел.
# Используем списковое включение (list comprehension) для краткости:
# для каждого 'literal' в генераторе вызываем .toPython(),
# чтобы получить чистое int-значение, и добавляем его в список.
msu_student_size = [literal.toPython() for literal in results_generator]

print("Найденная численность студентов МГУ (используя SequencePath):")
print(msu_student_size)

Найденная численность студентов МГУ (используя SequencePath):
[40000, 38000, 42000]


<p class="task" id="6"></p>

6\. Расширьте имеющийся граф путем добавления информации об университетах из инфобоксов на странице Википедии. Для расширения графа используйте возможности метода `Graph.add`.

Обратите внимание на следующие моменты:
* для некоторых строковых значений указан язык (например, `motto` и `motto_lang`). В таком случае язык должен быть указан как языковой тег;
* числовые значения могут содержать дополнительный разделитель (`,`), его нужно убрать перед добавлением тройки;
* у числовых типов должен быть указан URI типа;
* если не указано иное, считаем, что числовая характеристика представлена на 2024 год;
* моменты, на которые информация актуальна, должны быть указаны в виде квалификатора; материализацию производите по схеме 2 (см. занятие __"RDF. Сериализация RDF-графа в форматы N-Triples и Turtle."__).

Для получения информации из инфобоксов можно воспользоваться любым готовым решением, например, [wptools](https://pypi.org/project/wptools/). Если это удобно, можно получать информацию из инфобоксов на английском языке.


- [ ] Проверено на семинаре

In [31]:
pip install wptools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 2.7 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 2.9 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [wptools]m3/4 [wptools]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import wptools 
import re      
from rdflib import Graph, Namespace, Literal, XSD, URIRef
import uuid    

In [ ]:
# Используем наш существующий граф g_univ и пространства имен
E = Namespace("http://www.fa.ru/entity/")
P = Namespace("http://www.fa.ru/prop/")
PD = Namespace("http://www.fa.ru/prop/direct/")
PS = Namespace("http://www.fa.ru/prop/statement/")
PQ = Namespace("http://www.fa.ru/prop/qualifier/")

In [ ]:
# Создадим словарь, чтобы сопоставить URI из нашего графа с названиями статей в англоязычной Википедии 
wiki_pages = {
    E.MoscowStateUniversity: "Moscow State University",
    E.BaumanMoscowStateTechnicalUniversity: "Bauman Moscow State Technical University",
    E.SaintPetersburgStateUniversity: "Saint Petersburg State University",
    E.SpbPolytechnicUniversity: "Peter the Great St. Petersburg Polytechnic University"
}

print(f"Начальное количество троек в графе: {len(g_univ)}")

Начальное количество троек в графе: 52


In [9]:
# Пройдемся по каждому университету из нашего словаря
for uni_uri, page_title in wiki_pages.items():
    print(f"\n---> Обработка: {page_title}")
    try:
        page = wptools.page(page_title, lang='en').get_parse()
        infobox = page.data.get('infobox', {}) # Извлекаем инфобокс
        if not infobox:
            print(f"    [!] Инфобокс не найден для {page_title}")
            continue
        if 'motto' in infobox:
            motto_text = infobox.get('motto')
            motto_lang = infobox.get('motto_lang', 'en') # По умолчанию - английский
            # Создаем литерал с указанием языка
            motto_literal = Literal(motto_text, lang=motto_lang)
            # Добавляем тройку в граф
            g_univ.add((uni_uri, PD.motto, motto_literal))
            print(f"    + Добавлен девиз: '{motto_text}'@{motto_lang}")

        # Словарь для удобной обработки однотипных полей
        numeric_fields_to_process = {
            'students': (P.studentSize, PS.studentSize),
            'academic_staff': (P.staffSize, PS.staffSize) # Придумаем новые предикаты
        }

        for field, (prop, prop_statement) in numeric_fields_to_process.items():
            if field in infobox:
                # 1. Получаем и очищаем значение
                raw_value = infobox.get(field)
                # Удаляем запятые, сноски [1], и берем только цифры
                cleaned_value = re.sub(r'\[.*?\]', '', str(raw_value)) # Удаляем [1], [2]..
                cleaned_value = re.sub(r',', '', cleaned_value)         # Удаляем ,
                numeric_value = int(re.search(r'\d+', cleaned_value).group()) # Ищем первое число

                # 2. Создаем литерал с правильным типом данных
                value_literal = Literal(numeric_value, datatype=XSD.integer)

                # 3. Реализуем "Схему 2" (материализация)
                # Создаем уникальный URI для нового узла-утверждения (statement)
                statement_uri = E[f"statement/{uuid.uuid4()}"]

                # 3.1. Связываем университет с узлом-утверждением
                g_univ.add((uni_uri, prop, statement_uri))

                # 3.2. Добавляем само значение к узлу-утверждению
                g_univ.add((statement_uri, prop_statement, value_literal))

                # 3.3. Добавляем квалификатор - год (по условию - 2024)
                year_literal = Literal("2024", datatype=XSD.gYear)
                g_univ.add((statement_uri, PQ.atYear, year_literal))
                
                print(f"    + Добавлено '{field}': {numeric_value} (на 2024 год)")

    except Exception as e:
        print(f"    [!] Произошла ошибка при обработке {page_title}: {e}")


---> Обработка: Moscow State University


en.wikipedia.org (parse) Moscow State University
en.wikipedia.org (imageinfo) File:МГУ, вид с воздуха.jpg
Moscow State University (en) data
{
  image: <list(1)> {'kind': 'parse-image', 'file': 'File:МГУ, вид ...
  infobox: <dict(25)> name, native_name, native_name_lang, former_...
  iwlinks: <list(13)> https://commons.wikimedia.org/wiki/Category:...
  pageid: 374544
  parsetree: <str(73650)> <root><template><title>short description...
  requests: <list(2)> parse, imageinfo
  title: Moscow State University
  wikibase: Q13164
  wikidata_url: https://www.wikidata.org/wiki/Q13164
  wikitext: <str(55110)> {{short description|Public research unive...
}
en.wikipedia.org (parse) Bauman Moscow State Technical University


    + Добавлен девиз: 'Наука есть ясное познание истины, просвещение разума'@ru
    + Добавлено 'students': 39282 (на 2024 год)
    + Добавлено 'academic_staff': 5000 (на 2024 год)

---> Обработка: Bauman Moscow State Technical University


en.wikipedia.org (imageinfo) File:МГТУ им.Баумана, главное здание...
Bauman Moscow State Technical University (en) data
{
  image: <list(1)> {'kind': 'parse-image', 'file': 'File:МГТУ им.Б...
  infobox: <dict(18)> name, native_name, image, motto, established...
  iwlinks: <list(3)> https://commons.wikimedia.org/wiki/Category:B...
  pageid: 714503
  parsetree: <str(41178)> <root><template><title>Short description...
  requests: <list(2)> parse, imageinfo
  title: Bauman Moscow State Technical University
  wikibase: Q1472245
  wikidata_url: https://www.wikidata.org/wiki/Q1472245
  wikitext: <str(31930)> {{Short description|Public technical univ...
}
en.wikipedia.org (parse) Saint Petersburg State University


    + Добавлен девиз: '«'''М'''ужество, '''В'''оля, '''Т'''руд, '''У'''порство!»<br />"Courage, will, labor, perseverance!"'@en
    + Добавлено 'students': 19000 (на 2024 год)

---> Обработка: Saint Petersburg State University


Saint Petersburg State University (en) data
{
  infobox: <dict(24)> name, native_name, former_names, image_name,...
  iwlinks: <list(15)> https://commons.wikimedia.org/wiki/Category:...
  pageid: 649879
  parsetree: <str(62612)> <root><template><title>Short description...
  requests: <list(1)> parse
  title: Saint Petersburg State University
  wikibase: Q27621
  wikidata_url: https://www.wikidata.org/wiki/Q27621
  wikitext: <str(51116)> {{Short description|Public research unive...
}
en.wikipedia.org (parse) Peter the Great St. Petersburg Polytechn...


    + Добавлен девиз: '''Hic tuta perennat'' ([[Latin]])'@en

---> Обработка: Peter the Great St. Petersburg Polytechnic University
    + Добавлено 'students': 32250 (на 2024 год)


Peter the Great St. Petersburg Polytechnic University (en) data
{
  infobox: <dict(19)> name, former_name, native_name, established,...
  iwlinks: <list(1)> https://ru.wikipedia.org/wiki/%D0%92%D0%BE%D0...
  pageid: 6258161
  parsetree: <str(34235)> <root><template><title>Short description...
  requests: <list(1)> parse
  title: Peter the Great St. Petersburg Polytechnic University
  wikibase: Q1379834
  wikidata_url: https://www.wikidata.org/wiki/Q1379834
  wikitext: <str(23549)> {{Short description|Technical university ...
}


In [ ]:
print(f"Итоговое количество троек в графе: {len(g_univ)}")

Итоговое количество троек в графе: 79
Граф успешно расширен данными из Википедии.


<p class="task" id="7"></p>

7\. Сериализуйте расширенный граф в формат `turtle` и сохраните сериализованное представление на диск. Считайте загруженный файл и выведите на экран количество троек.

Выведите на экран всю числовую информацию об университетах, актуальную на 2024 год. В выводе должно присутствовать название университета, URI (или название) отношения и само значение (численность, количество сотрудников и т.д.). Для поиска используйте любые методы графа, не требующие работы со SPARQL. 


- [ ] Проверено на семинаре

In [ ]:
# Используем пространства имен, определенные в предыдущих задачах
E = Namespace("http://www.fa.ru/entity/")
P = Namespace("http://www.fa.ru/prop/")
PD = Namespace("http://www.fa.ru/prop/direct/")
PS = Namespace("http://www.fa.ru/prop/statement/")
PQ = Namespace("http://www.fa.ru/prop/qualifier/")

output_filename = "university_extended.ttl"

In [12]:
g_univ.serialize(destination=output_filename, format="turtle")
print(f"Граф успешно сериализован и сохранен в файл: '{output_filename}'")

Граф успешно сериализован и сохранен в файл: 'university_extended.ttl'


In [ ]:
g_loaded = Graph()

g_loaded.parse(output_filename, format="turtle")
print(f"Граф успешно загружен из файла '{output_filename}'.")
# Выводим количество троек, чтобы убедиться, что все на месте.
print(f"Количество троек в загруженном графе: {len(g_loaded)}")

Граф успешно загружен из файла 'university_extended.ttl'.
Количество троек в загруженном графе: 79


In [14]:
# Создаем литерал для 2024 года, который будем искать в графе
year_to_find = Literal("2024", datatype=XSD.gYear)

statements_2024 = g_loaded.subjects(predicate=PQ.atYear, object=year_to_find)

for stmt_uri in statements_2024:
    
    university_name = "Не найден"
    relation_name = "Не найден"
    value = "Не найдено"

    # Найти университет, к которому относится это утверждение.
    for uni_uri, prop in g_loaded.subject_predicates(object=stmt_uri):
        # Очищаем URI университета до простого названия
        university_name = str(uni_uri).replace(str(E), "")
        # Очищаем URI предиката до названия отношения
        relation_name = str(prop).replace(str(P), "")
        
    # Найти само числовое значение, хранящееся в утверждении.
    for prop, val_literal in g_loaded.predicate_objects(subject=stmt_uri):
        if str(prop).startswith(str(PS)):
            value = val_literal.toPython() # Преобразуем литерал в число

    print(f"- Университет: {university_name}, Отношение: {relation_name}, Значение: {value}")

- Университет: BaumanMoscowStateTechnicalUniversity, Отношение: studentSize, Значение: 19000
- Университет: MoscowStateUniversity, Отношение: staffSize, Значение: 5000
- Университет: MoscowStateUniversity, Отношение: studentSize, Значение: 39282
- Университет: SpbPolytechnicUniversity, Отношение: studentSize, Значение: 32250
- Университет: MoscowStateUniversity, Отношение: studentSize, Значение: 39282
- Университет: SpbPolytechnicUniversity, Отношение: studentSize, Значение: 32250
- Университет: MoscowStateUniversity, Отношение: staffSize, Значение: 5000
- Университет: BaumanMoscowStateTechnicalUniversity, Отношение: studentSize, Значение: 19000
